## Flight data

<a href="#PCA" style="text-decoration: none;">
    <button style="padding:10px; font-size:15px;">PCA</button>
</a>

<a href="#PC" style="text-decoration: none;">
    <button style="padding:10px; font-size:15px;">PCA Components</button>
</a>

<a href="#Clusters" style="text-decoration: none;">
    <button style="padding:10px; font-size:15px;">Clusters</button>
</a>

<a href="#Cluster-Visual" style="text-decoration: none;">
    <button style="padding:10px; font-size:15px;">Cluster Visual</button>
</a>

<a href="#Time-Series" style="text-decoration: none;">
    <button style="padding:10px; font-size:15px;">Time Series</button>
</a>

## Data Prep

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
L = [' PHX',' LAX',' SAN',' SFO',' DEN',' FLL',' MIA',' MCO',' TPA',' ATL',
     ' MDW',' ORD',' BWI',' BOS',' DTW',' MSP',' LAS',' EWR',' JFK',' LGA',
     ' CLT',' PHL',' BNA',' AUS',' DFW',' IAH',' SLC',' DCA',' IAD',' SEA',' MCI',' IND',' HNL',' ANC']
flight = pd.read_csv('/Users/jing/Documents/Flight Data/DATA/Cancelled2024-2025.8.CSV')
#scheduled commercial flights with dep, arr, and air delays
#airportLocation.to_csv('airportLocation.csv')

In [3]:
wrong = ['FCA','AZA','FWH','USA','CBD',
         'CLD','QQT','XXW','GAL','SCE',
         'BKC','RDB','BKG','SPN','HAR',
         'HHH','UST']
correct = ['GPI','IWA','NFW','JQF','PACD',
          'CRQ','FFC','LCQ','PAGA','UNV',
           'PABL','DGG','BBG','GSN','CXY',
           'HXD','SGJ']
airport = pd.read_csv('../../GDP/AirportLoc/Airport_Loc.csv')
airport = airport[airport['AIRPORT_COUNTRY_CODE_ISO']=='US']
airport = airport[airport['AIRPORT_THRU_DATE'].isna()]
airport['AIRPORT'] = airport['AIRPORT'].replace(wrong,correct)
airport = airport[['AIRPORT_STATE_CODE','LATITUDE','LONGITUDE','AIRPORT']].head(-1)
airport = airport[~airport['AIRPORT_STATE_CODE'].isin(['PR','TT','VI'])]

In [4]:
airport['LOCID'] = ' '+airport['AIRPORT']

In [5]:
Airports = list(set(flight['DEP_LOCID'].unique())|set(flight['ARR_LOCID'].unique()))
len(Airports)#521
#Number of all U.S. airports exsits in ASPM excluding PR, TT, and VI

747

In [6]:
Airports = [x for x in Airports if x in list(airport[airport['LOCID'].isin(Airports)]['LOCID'])]#490

In [7]:
len(Airports)

464

In [8]:
Airports_temp = Airports.copy()
Airports_temp.append('PAGA')
Airports_temp.append('PAKT')
Airports_temp.append('PAVD')
Airports_temp.append('PAIL')
Airports_temp.append('PACD')
Airports_temp.append('PASN')

In [9]:
len(flight)#53103315

426024

In [10]:
flight = flight[(flight['DEP_LOCID'].isin(Airports_temp))&(flight['ARR_LOCID'].isin(Airports_temp))]

In [11]:
len(flight)#91134360
#Number of flights OD pairs all within U.S.

391313

In [12]:
airport = airport[airport['LOCID'].isin(Airports)]
len(airport)#all flights in airport 490

464

In [13]:
center = pd.read_csv('../../GDP/AirportLoc/US_center_apt.CSV').drop('Unnamed: 0',axis=1)
centers = []
for i in range(len(center)):
    for j in range(len(eval(center.iloc[i]['AIRPORT']))):
        centers.append([center.iloc[i]['Con_Zone'],' '+eval(center.iloc[i]['AIRPORT'])[j]])
centers = pd.DataFrame(centers,columns = ['Zone','Airport'])#ZAN, ZHN
#get ARTCC center excluding ZAN and ZHN

In [14]:
wrong = [' AZA',
 ' BKG',
 #' DFW',#RRO TX
 ' CLD',
 ' FCA',
 ' FWH',
 ' HAR',
 ' HHH',
 #' DFW',#HRL TX
 #' DFW',#LRD TX
 #' DFW',#MFE TX
 ' QQT',
 #' PHL',#UNV PA
 ' SGJ',
 ' XXW']
correct = [' IWA',
 ' BBG',
 #' BRO',
 ' CRQ',
 ' GPI',
 ' NFW',
 ' CXY',
 ' HXD',
 #' HRL',
 #' LRD',
 #' MFE',
 ' FFC',
 #' UNV',
 ' SGJ',
 ' LCQ']
centers = centers.replace(wrong,correct)

In [15]:
centers_append = pd.DataFrame(columns = ['Zone','Airport'])
centers_append['Zone'] = ['ZFW','ZFW','ZFW','ZFW','ZDC','ZFW','ZOA']
centers_append['Airport'] = [' RRO',' HRL',' LRD',' MFE',' UNV',' BRO',' SGJ']
centers = pd.concat([centers,centers_append])
#Adjust data

In [16]:
airport = airport.merge(centers,left_on = 'LOCID',right_on = 'Airport',how = 'left')
len(airport[airport['AIRPORT_STATE_CODE'].isin(['HI','AK'])])

35

In [17]:
airport['Zone'] = airport.apply(lambda row: 'ZHN' if row.AIRPORT_STATE_CODE=='HI' else 'ZAN' if row.AIRPORT_STATE_CODE=='AK' else row.Zone,axis=1)

In [18]:
airport = airport[['AIRPORT_STATE_CODE', 'LATITUDE', 'LONGITUDE', 'AIRPORT', 'LOCID',
       'Zone']]

In [19]:
airport.loc[490] = ['AK',0,0,'PAGA','PAGA','ZAN']
airport.loc[491] = ['AK',0,0,'PAKT','PAKT','ZAN']
airport.loc[492] = ['AK',0,0,'PAVD','PAVD','ZAN']
airport.loc[493] = ['AK',0,0,'PAIL','PAIL','ZAN']
airport.loc[494] = ['AK',0,0,'PACD','PACD','ZAN']
airport.loc[495] = ['AK',0,0,'PASN','PASN','ZAN']

In [20]:
airport

,AIRPORT_STATE_CODE,LATITUDE,LONGITUDE,AIRPORT,LOCID,Zone
0,PA,40.652222,-75.440556,ABE,ABE,ZNY
1,TX,32.411389,-99.681944,ABI,ABI,ZFW
2,NM,35.038889,-106.608333,ABQ,ABQ,ZAB
3,SD,45.446667,-98.422500,ABR,ABR,ZMP
4,GA,31.535556,-84.194444,ABY,ABY,ZJX
...,...,...,...,...,...,...
491,AK,0.000000,0.000000,PAKT,PAKT,ZAN
492,AK,0.000000,0.000000,PAVD,PAVD,ZAN
493,AK,0.000000,0.000000,PAIL,PAIL,ZAN
494,AK,0.000000,0.000000,PACD,PACD,ZAN


In [21]:
#all airports in ASPM dataset with their location and Zone

In [22]:
centerAirports = centers[centers['Airport'].isin(L)]
countZone = centerAirports.groupby('Zone').count().reset_index()
singleZones = list(countZone[countZone['Airport']==1]['Zone'])
singleZones.append('ZHN')
singleZones.append('ZAN')
multiZones = list(countZone[countZone['Airport']!=1]['Zone'])

In [23]:
import math

def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in kilometers
    R = 6371.0  
    
    # Convert latitude and longitude from degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Differences in coordinates
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in kilometers
    distance = R * c
    
    return distance

In [24]:
airport

,AIRPORT_STATE_CODE,LATITUDE,LONGITUDE,AIRPORT,LOCID,Zone
0,PA,40.652222,-75.440556,ABE,ABE,ZNY
1,TX,32.411389,-99.681944,ABI,ABI,ZFW
2,NM,35.038889,-106.608333,ABQ,ABQ,ZAB
3,SD,45.446667,-98.422500,ABR,ABR,ZMP
4,GA,31.535556,-84.194444,ABY,ABY,ZJX
...,...,...,...,...,...,...
491,AK,0.000000,0.000000,PAKT,PAKT,ZAN
492,AK,0.000000,0.000000,PAVD,PAVD,ZAN
493,AK,0.000000,0.000000,PAIL,PAIL,ZAN
494,AK,0.000000,0.000000,PACD,PACD,ZAN


In [25]:
huborzone = []
for i in range(len(airport)):
    if airport.iloc[i]['Zone'] in singleZones:
        huborzone.append(airport.iloc[i]['Zone'])
    else:
        temp = airport[(airport['LOCID'].isin(L))&(airport['Zone']==airport.iloc[i]['Zone'])]
        lat = airport.iloc[i]['LATITUDE']
        long = airport.iloc[i]['LONGITUDE']
        temp['dis'] = temp.apply(lambda row: haversine(lat,long,row.LATITUDE,row.LONGITUDE), axis=1)
        huborzone.append(temp.iloc[np.argmin(temp['dis'])]['LOCID'])

In [26]:
airport['huborzone'] = huborzone

In [27]:
## Airport Grouping according to ARTCC/large hub
## ref. TC1_Jing.pptx,Page 10
airport.groupby(['Zone','huborzone']).count()

AIRPORT_STATE_CODE  LATITUDE  LONGITUDE  AIRPORT  LOCID
Zone huborzone                                                         
ZAB  ZAB                        16        16         16       16     16
ZAN  ZAN                        32        32         32       32     32
ZAU   MDW                       11        11         11       11     11
      ORD                       11        11         11       11     11
ZBW  ZBW                        23        23         23       23     23
ZDC   BWI                        3         3          3        3      3
      DCA                       10        10         10       10     10
      IAD                        9         9          9        9      9
      PHL                        4         4          4        4      4
ZDV  ZDV                        24        24         24       24     24
ZFW  ZFW                        23        23         23       23     23
ZHN  ZHN                         9         9          9        9      9
ZHU   AUS                        5         5          5        5      5
      IAH                       17        17         17       17     17
ZID  ZID                        15        15         15       15     15
ZJX   MCO                       15        15         15       15     15
      TPA                        8         8          8        8      8
ZKC  ZKC                        26        26         26       26     26
ZLA   LAS                        4         4          4        4      4
      LAX                       12        12         12       12     12
      SAN                        5         5          5        5      5
ZLC  ZLC                        26        26         26       26     26
ZMA   FLL                        9         9          9        9      9
      MIA                        4         4          4        4      4
ZME  ZME                        23        23         23       23     23
ZMP  ZMP                        39        39         39       39     39
ZNY   EWR                       10        10         10       10     10
      JFK                        2         2          2        2      2
      LGA                        2         2          2        2      2
ZOA  ZOA                        13        13         13       13     13
ZOB  ZOB                        21        21         21       21     21
ZSE  ZSE                        20        20         20       20     20
ZTL   ATL                       11        11         11       11     11
      CLT                        8         8          8        8      8

In [28]:
airportLocation = airport.copy()

In [29]:
airport = airport[['LOCID','huborzone']]
airport.columns = ['ARR_LOCID','huborzone']
airport

,ARR_LOCID,huborzone
0,ABE,EWR
1,ABI,ZFW
2,ABQ,ZAB
3,ABR,ZMP
4,ABY,TPA
...,...,...
491,PAKT,ZAN
492,PAVD,ZAN
493,PAIL,ZAN
494,PACD,ZAN


In [30]:
len(flight)

391313

In [31]:
flight = flight.merge(airport,on = 'ARR_LOCID')

In [32]:
huborzone_temp = airport['huborzone'].unique()

In [33]:
huborzone_temp

array([' EWR', 'ZFW', 'ZAB', 'ZMP', ' TPA', 'ZBW', 'ZSE', ' PHL', 'ZAN',
       ' DCA', ' IAH', ' CLT', ' ATL', ' ORD', 'ZOB', 'ZDV', ' AUS',
       ' MDW', ' LAX', 'ZOA', 'ZLC', 'ZME', 'ZID', 'ZKC', ' MCO', ' BWI',
       ' IAD', ' MIA', ' FLL', ' JFK', 'ZHN', ' LAS', ' SAN', ' LGA'],
      dtype=object)

In [34]:
flight['FAACARRIER'].value_counts()

AAL    62178
UAL    47077
SWA    40252
SKW    34802
DAL    31276
JIA    24124
RPA    23364
EDV    16559
ENY    15605
NKS    13531
ASA    12194
FFT    11330
JBU    10700
UPS     6818
FDX     6779
ASH     6304
UCA     5534
PDT     4602
GJS     4329
AAY     2911
FDY     2895
QXE     2778
AWI     2044
HAL     1704
VTE     1149
SIL      134
SCX      127
BTQ       92
ACA       64
WSN       28
SRY       21
GUN        3
ASQ        2
QTR        1
CPZ        1
GPD        1
Name: FAACARRIER, dtype: int64

In [35]:
flight

,FAACARRIER,FLTNO,DEP_LOCID,ARR_LOCID,DEP_YYYYMM,DEP_DAY,ARR_YYYYMM,ARR_DAY,DEP_SECS,D_DEP_SECS,...,ARR_SECS,D_ARR_SECS,T_ARR_SECS,EQUIP,EQUIP_CLASS,CANCCODE,MAXCAUSE,ADD_FLAG,REC_TYPE,huborzone
0,AAL,28,LAX,JFK,202401,13,202401,14,1389682320,01/13/2024,...,1389701820,01/14/2024,07:17,----,-,,0,0,R,JFK
1,AAL,374,CLT,JFK,202401,9,202401,9,1389293460,01/09/2024,...,1389300060,01/09/2024,15:41,,,B,0,1,A,JFK
2,AAL,374,CLT,JFK,202401,9,202401,9,1389293460,01/09/2024,...,1389300060,01/09/2024,15:41,B738,J,,0,0,R,JFK
3,AAL,411,EGE,JFK,202401,5,202401,5,1388947560,01/05/2024,...,1388961600,01/05/2024,17:40,A319,J,,0,0,R,JFK
4,AAL,411,EGE,JFK,202401,6,202401,6,1389035640,01/06/2024,...,1389047700,01/06/2024,17:35,----,-,,0,0,R,JFK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391308,SWA,1,BOI,MYL,202505,12,202505,12,1431541980,05/12/2025,...,1431543660,05/12/2025,13:01,B737,J,,0,0,R,ZLC
391309,DAL,8878,CLE,BKL,202507,20,202507,20,1437525000,07/20/2025,...,1437525900,07/20/2025,20:45,----,-,,0,1,R,ZOB
391310,DAL,3022,ATL,FFC,202508,12,202508,12,1439462160,08/12/2025,...,1439462760,08/12/2025,06:46,B739,J,,0,1,R,ATL
391311,NKS,2457,FLL,FXE,202508,18,202508,18,1440002760,08/18/2025,...,1440003360,08/18/2025,12:56,----,-,,0,1,R,FLL


In [36]:
count = flight.groupby(['DEP_YYYYMM','DEP_DAY','huborzone']).agg({'FLTNO':'count'}).reset_index()

In [37]:
count = count[((count['DEP_YYYYMM']>=202106)|(count['DEP_YYYYMM']<202003))]
#count = count[count['DEP_YYYYMM']>201600]

In [38]:
count1= flight.groupby(['DEP_YYYYMM','DEP_DAY']).agg({'FLTNO':'count'}).reset_index()
count1 = count1[((count1['DEP_YYYYMM']>=202106)|(count1['DEP_YYYYMM']<202003))]

In [39]:
count1

,DEP_YYYYMM,DEP_DAY,FLTNO
0,202401,1,87
1,202401,2,124
2,202401,3,118
3,202401,4,190
4,202401,5,112
...,...,...,...
604,202508,27,124
605,202508,28,459
606,202508,29,258
607,202508,30,482


In [40]:
empty = []
for i in np.unique(count[['DEP_YYYYMM','DEP_DAY']], axis=0):
    for j in range(len(huborzone_temp)):
        empty.append([i[0],i[1],huborzone_temp[j]])
empty = pd.DataFrame(empty,columns = ['DEP_YYYYMM','DEP_DAY','huborzone'])
len(empty)#34*4869

20706

In [41]:
full = empty.merge(count,on = ['DEP_YYYYMM','DEP_DAY','huborzone'],how = 'left')
full = full.fillna(0)
full

,DEP_YYYYMM,DEP_DAY,huborzone,FLTNO
0,202401,1,EWR,1.0
1,202401,1,ZFW,7.0
2,202401,1,ZAB,5.0
3,202401,1,ZMP,2.0
4,202401,1,TPA,0.0
...,...,...,...,...
20701,202508,31,JFK,3.0
20702,202508,31,ZHN,0.0
20703,202508,31,LAS,4.0
20704,202508,31,SAN,3.0


In [43]:
## Get final 2678 day rows and 2+4*34 feature columns DataFrame

In [42]:
final = []
for i in range(609):
    temp = []
    temp.append(count1.iloc[i]['DEP_YYYYMM'])
    temp.append(count1.iloc[i]['DEP_DAY'])
    for j in range(34):
        temp.append(full.iloc[i*34+j]['FLTNO'])
    final.append(temp)

In [43]:
final = pd.DataFrame(final)

In [44]:
final

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,202401,1,1.0,7.0,5.0,2.0,0.0,1.0,29.0,0.0,...,0.0,2.0,1.0,0.0,2.0,2.0,0.0,2.0,3.0,0.0
1,202401,2,2.0,7.0,5.0,0.0,0.0,1.0,25.0,1.0,...,5.0,1.0,2.0,0.0,1.0,2.0,1.0,5.0,1.0,0.0
2,202401,3,3.0,9.0,4.0,2.0,0.0,3.0,10.0,0.0,...,5.0,2.0,1.0,1.0,1.0,6.0,4.0,1.0,1.0,0.0
3,202401,4,2.0,44.0,32.0,1.0,0.0,0.0,8.0,1.0,...,4.0,1.0,0.0,1.0,0.0,0.0,14.0,6.0,1.0,0.0
4,202401,5,2.0,12.0,6.0,0.0,1.0,0.0,12.0,2.0,...,1.0,0.0,1.0,0.0,1.0,3.0,3.0,4.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604,202508,27,8.0,6.0,3.0,1.0,0.0,6.0,3.0,6.0,...,4.0,0.0,3.0,1.0,2.0,1.0,1.0,2.0,2.0,1.0
605,202508,28,9.0,170.0,16.0,6.0,5.0,4.0,7.0,2.0,...,8.0,0.0,6.0,5.0,4.0,1.0,4.0,4.0,6.0,1.0
606,202508,29,7.0,77.0,14.0,3.0,4.0,2.0,4.0,7.0,...,4.0,1.0,2.0,11.0,3.0,1.0,4.0,2.0,2.0,4.0
607,202508,30,4.0,200.0,16.0,9.0,9.0,6.0,14.0,6.0,...,10.0,4.0,7.0,6.0,7.0,2.0,1.0,6.0,5.0,3.0


In [45]:
final[final[0]==202407]

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
182,202407,1,61.0,31.0,11.0,9.0,4.0,27.0,18.0,121.0,...,29.0,2.0,25.0,23.0,20.0,24.0,4.0,13.0,11.0,46.0
183,202407,2,14.0,11.0,5.0,6.0,0.0,8.0,6.0,4.0,...,8.0,0.0,8.0,2.0,3.0,4.0,9.0,4.0,0.0,4.0
184,202407,3,3.0,8.0,7.0,2.0,3.0,9.0,8.0,4.0,...,2.0,1.0,3.0,5.0,0.0,1.0,3.0,4.0,3.0,2.0
185,202407,4,8.0,8.0,4.0,4.0,0.0,5.0,4.0,1.0,...,0.0,1.0,9.0,2.0,0.0,1.0,5.0,3.0,4.0,2.0
186,202407,5,12.0,13.0,10.0,3.0,0.0,22.0,11.0,12.0,...,12.0,4.0,11.0,6.0,2.0,8.0,8.0,0.0,15.0,36.0
187,202407,6,17.0,41.0,23.0,8.0,7.0,7.0,24.0,7.0,...,13.0,3.0,8.0,6.0,5.0,2.0,12.0,6.0,7.0,21.0
188,202407,7,9.0,94.0,32.0,21.0,17.0,43.0,19.0,21.0,...,27.0,10.0,20.0,6.0,8.0,14.0,18.0,18.0,9.0,30.0
189,202407,8,32.0,282.0,81.0,27.0,36.0,35.0,20.0,29.0,...,89.0,26.0,37.0,29.0,27.0,4.0,11.0,42.0,30.0,34.0
190,202407,9,9.0,104.0,28.0,12.0,17.0,7.0,20.0,19.0,...,24.0,11.0,20.0,23.0,9.0,3.0,7.0,16.0,12.0,14.0
191,202407,10,36.0,21.0,15.0,7.0,6.0,56.0,25.0,31.0,...,26.0,2.0,33.0,6.0,2.0,48.0,3.0,22.0,8.0,80.0


In [46]:
final[final[0]>202400].to_csv('Cancelled_34_24_25.csv')

In [47]:
#final.to_csv('Cancelled_34.csv')

In [48]:
final

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,202401,1,1.0,7.0,5.0,2.0,0.0,1.0,29.0,0.0,...,0.0,2.0,1.0,0.0,2.0,2.0,0.0,2.0,3.0,0.0
1,202401,2,2.0,7.0,5.0,0.0,0.0,1.0,25.0,1.0,...,5.0,1.0,2.0,0.0,1.0,2.0,1.0,5.0,1.0,0.0
2,202401,3,3.0,9.0,4.0,2.0,0.0,3.0,10.0,0.0,...,5.0,2.0,1.0,1.0,1.0,6.0,4.0,1.0,1.0,0.0
3,202401,4,2.0,44.0,32.0,1.0,0.0,0.0,8.0,1.0,...,4.0,1.0,0.0,1.0,0.0,0.0,14.0,6.0,1.0,0.0
4,202401,5,2.0,12.0,6.0,0.0,1.0,0.0,12.0,2.0,...,1.0,0.0,1.0,0.0,1.0,3.0,3.0,4.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604,202508,27,8.0,6.0,3.0,1.0,0.0,6.0,3.0,6.0,...,4.0,0.0,3.0,1.0,2.0,1.0,1.0,2.0,2.0,1.0
605,202508,28,9.0,170.0,16.0,6.0,5.0,4.0,7.0,2.0,...,8.0,0.0,6.0,5.0,4.0,1.0,4.0,4.0,6.0,1.0
606,202508,29,7.0,77.0,14.0,3.0,4.0,2.0,4.0,7.0,...,4.0,1.0,2.0,11.0,3.0,1.0,4.0,2.0,2.0,4.0
607,202508,30,4.0,200.0,16.0,9.0,9.0,6.0,14.0,6.0,...,10.0,4.0,7.0,6.0,7.0,2.0,1.0,6.0,5.0,3.0


In [49]:
huborzone_temp

array([' EWR', 'ZFW', 'ZAB', 'ZMP', ' TPA', 'ZBW', 'ZSE', ' PHL', 'ZAN',
       ' DCA', ' IAH', ' CLT', ' ATL', 'ZDV', ' ORD', 'ZOB', ' MIA',
       ' AUS', ' MDW', ' LAX', ' LAS', 'ZOA', 'ZLC', 'ZME', 'ZID', 'ZKC',
       ' MCO', ' BWI', ' IAD', ' SAN', ' FLL', 'ZHN', ' JFK', ' LGA'],
      dtype=object)

In [50]:
[' EWR', 'ZFW', 'ZAB', 'ZMP', ' TPA', 'ZBW', 'ZSE', ' PHL', 'ZAN',
       ' DCA', ' IAH', ' CLT', ' ATL', 'ZDV', ' ORD', 'ZOB', ' MIA',
       ' AUS', ' MDW', ' LAX', ' LAS', 'ZOA', 'ZLC', 'ZME', 'ZID', 'ZKC',
       ' MCO', ' BWI', ' IAD', ' SAN', ' FLL', 'ZHN', ' JFK', ' LGA']

[' EWR',
 'ZFW',
 'ZAB',
 'ZMP',
 ' TPA',
 'ZBW',
 'ZSE',
 ' PHL',
 'ZAN',
 ' DCA',
 ' IAH',
 ' CLT',
 ' ATL',
 'ZDV',
 ' ORD',
 'ZOB',
 ' MIA',
 ' AUS',
 ' MDW',
 ' LAX',
 ' LAS',
 'ZOA',
 'ZLC',
 'ZME',
 'ZID',
 'ZKC',
 ' MCO',
 ' BWI',
 ' IAD',
 ' SAN',
 ' FLL',
 'ZHN',
 ' JFK',
 ' LGA']

In [51]:
flight

,Unnamed: 0,FAACARRIER,FLTNO,DEP_LOCID,ARR_LOCID,DEP_YYYYMM,DEP_DAY,ARR_YYYYMM,ARR_DAY,DEP_SECS,...,ARR_SECS,D_ARR_SECS,T_ARR_SECS,EQUIP,EQUIP_CLASS,CANCCODE,MAXCAUSE,ADD_FLAG,REC_TYPE,huborzone
0,0,AAH,224,HNL,OGG,201001,12,201001,12,947775960,...,947777580,01/12/2010,05:33,B732,J,,0,0,R,ZHN
1,34,AAL,253,LAX,OGG,201001,11,201001,11,947698200,...,947718900,01/11/2010,13:15,,,A,0,1,A,ZHN
2,35,AAL,253,LAX,OGG,201001,11,201001,11,947698200,...,947718900,01/11/2010,13:15,B752,J,,0,0,R,ZHN
3,1552,ASA,857,PDX,OGG,201001,10,201001,10,947611500,...,947634000,01/10/2010,13:40,B738,J,,0,0,R,ZHN
4,1594,ASH,1039,HNL,OGG,201001,13,201001,13,947907300,...,947909400,01/13/2010,18:10,,,A,0,1,A,ZHN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4406405,4116842,FDY,583,HNL,LUP,202208,26,202208,26,1346027400,...,1346029440,08/26/2022,15:04,----,-,,0,1,R,ZHN
4406406,4435546,FDY,999,HNL,LUP,202308,21,202308,21,1377110700,...,1377113160,08/21/2023,09:26,----,-,,0,1,R,ZHN
4406407,4731097,FDY,583,HNL,LUP,202409,11,202409,11,1410572400,...,1410575400,09/11/2024,16:30,C208,T,,0,1,R,ZHN
4406408,4383792,DAL,8969,DTW,MUO,202302,28,202302,28,1362057900,...,1362071220,02/28/2023,10:07,----,-,,0,0,R,ZLC
